# Laboratorio 11 — Redes convolucionales con Fashion-MNIST
## Versión **estudiante**

**Curso:** Machine Learning (EIN143A25) · Paralelo 701 · USM Campus Concepción
**Fecha:** Miércoles 18 de noviembre de 2026 · Clase 11
**Bloque:** Lab · 11:50–13:35 (105 min) · Lab. Central 114, después de la teoría de la Clase 11
**Viene de:** la teoría «Clase 11 - CNN» (tensor → conectividad local → convolución → filtros → hiperparámetros → volúmenes → pooling → Fashion-MNIST → arquitectura → red en 3D → entrenar con aumento de datos → historia → transfer learning → aplicaciones).

| Parte | Tema | Tiempo sugerido | Diapositiva |
|---|---|---|---|
| **A** | Línea base: un MLP | 15 min | 13 |
| **B** | La primera CNN | 25 min | 14 |
| **C** | Mirar por dentro: filtros, mapas y ReLU | 20 min | 6 y 8 |
| **D** | Regularizar y aumentar datos | 20 min | 16 |
| **E** | Transferencia de aprendizaje (**opcional**, GPU recomendada) | 25 min | 19 |

## Objetivos de aprendizaje

1. Ver que un MLP aplana la imagen y una CNN la recorre con filtros, y que la CNN gana con la mitad de los parámetros.
2. Comprobar que las formas de los datos capa por capa son las de la diapositiva 14.
3. Ver que los filtros se aprenden solos y que sus mapas muestran la jerarquía: primero bordes, después partes.
4. Comprobar que regularizar y aumentar datos achica la brecha entre entrenamiento y validación.
5. (Opcional) Ver que una red preentrenada, con pocos datos, le gana a una CNN entrenada desde cero.

> **Cómo se trabaja.** Las celdas con `# TODO` se completan. Justo debajo de cada una hay una celda **Autochequeo**: si algo no está bien, falla con un mensaje que te dice qué revisar. Si no muestra error, vas bien. Las preguntas se responden en la celda de texto que sigue a cada una.

> Este laboratorio reutiliza el ciclo de entrenamiento de la Clase 10 (entropía cruzada, Adam, mini-lotes, Dropout, BatchNorm). La función `entrenar` ya viene lista en la celda de utilidades.

## Configuración

In [ ]:
import sys, math, time, warnings
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q scikit-learn pandas matplotlib   # torch y torchvision ya vienen en Colab

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score
from sklearn.model_selection import train_test_split

SEED = 42
RAPIDO = False   # True: divide las épocas a la mitad (solo para probar el notebook rápido)
T0 = time.time()

def set_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)

def ep(n):
    """Épocas efectivas (respeta RAPIDO)."""
    return max(1, n // 2) if RAPIDO else n

set_seed()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Paleta del deck de la Clase 11
AZUL, NARANJA, MARINO, GRIS, FONDO = "#2e69b2", "#ba6826", "#1b2740", "#8b929c", "#f6f6f4"
CMAP_DIV = LinearSegmentedColormap.from_list("azul_blanco_naranja", [AZUL, "#ffffff", NARANJA])  # negativo · 0 · positivo
CMAP_SEC = LinearSegmentedColormap.from_list("blanco_azul", ["#ffffff", AZUL, MARINO])           # 0 · alto
plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": "#fffefb", "axes.edgecolor": "#dcdad3",
    "axes.labelcolor": "#14181f", "text.color": "#14181f", "xtick.color": "#5d6675", "ytick.color": "#5d6675",
    "axes.prop_cycle": plt.cycler(color=[AZUL, NARANJA, MARINO, GRIS, "#9aa6b8"]),
    "axes.grid": True, "grid.color": "#e3e1da", "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 100,
})
print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · dispositivo: {DEVICE} · RAPIDO={RAPIDO}")

## Los datos: Fashion-MNIST

> **En palabras simples.** Para el computador, una foto en blanco y negro es solo una tabla de números: cada casilla dice qué tan oscuro es ese píxel. Fashion-MNIST tiene 70.000 fotos pequeñas de ropa (28 × 28 píxeles) de 10 tipos de prenda. Es el conjunto de la **diapositiva 13**.

- 60.000 fotos para entrenar (de ellas se apartan 6.000 para validación) y 10.000 de prueba.
- 10 prendas, 7.000 fotos de cada una: Camiseta, Pantalón, Suéter, Vestido, Abrigo, Sandalia, Camisa, Zapatilla, Bolso y Botín.

**Cómo se consiguen.** El cargador prueba tres caminos, en orden, y siempre dice cuál funcionó:

1. Descargar con `torchvision` (lo normal).
2. Si eso falla, bajar los 4 archivos desde un espejo de Google (`storage.googleapis.com`) a la carpeta `data/FashionMNIST/raw/`.
3. Solo si ambos fallan, un reemplazo de emergencia (`load_digits`) para que el notebook no se caiga. Si ves un aviso en MAYÚSCULAS, tus números no son los de Fashion-MNIST: avísale al profesor.

Revisa la línea que empieza con **«Conjunto:»** al final de la celda: debe decir Fashion-MNIST.

In [ ]:
CLASES_FASHION = ["Camiseta", "Pantalón", "Suéter", "Vestido", "Abrigo",
                  "Sandalia", "Camisa", "Zapatilla", "Bolso", "Botín"]
URL_ESPEJO = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/"
ARCHIVOS_FASHION = ["train-images-idx3-ubyte.gz", "train-labels-idx1-ubyte.gz",
                    "t10k-images-idx3-ubyte.gz", "t10k-labels-idx1-ubyte.gz"]

def fashion_torchvision(download=True):
    """Escalón 1: torchvision (descarga desde sus propios espejos si download=True)."""
    from torchvision import datasets
    return (datasets.FashionMNIST("data", train=True, download=download),
            datasets.FashionMNIST("data", train=False, download=download))

def fashion_desde_espejo():
    """Escalón 2: baja los 4 .gz del espejo de TensorFlow a data/FashionMNIST/raw/, los descomprime
    y vuelve a llamar a torchvision, que ya no necesita bajar nada."""
    import gzip, shutil, urllib.request
    from pathlib import Path
    raw = Path("data") / "FashionMNIST" / "raw"
    raw.mkdir(parents=True, exist_ok=True)
    for nombre in ARCHIVOS_FASHION:
        gz = raw / nombre
        if not gz.exists():
            print(f"  bajando {nombre} desde el espejo…")
            parcial = gz.with_suffix(".parcial")
            urllib.request.urlretrieve(URL_ESPEJO + nombre, parcial)
            parcial.rename(gz)   # solo queda con el nombre final si la descarga terminó
        with gzip.open(gz, "rb") as f_in, open(raw / nombre[:-3], "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
    return fashion_torchvision(download=False)

def reemplazo_digits():
    """Escalón 3 (emergencia): load_digits de scikit-learn, que viene sin descarga."""
    from sklearn.datasets import load_digits
    d = load_digits()
    X = torch.tensor(d.images / 16.0, dtype=torch.float32).unsqueeze(1)
    y = torch.tensor(d.target, dtype=torch.long)
    i_tr, i_te = train_test_split(np.arange(len(X)), test_size=0.15, stratify=d.target, random_state=SEED)
    i_tr, i_va = train_test_split(i_tr, test_size=0.15 / 0.85, stratify=d.target[i_tr], random_state=SEED)
    return dict(nombre="load_digits 8×8 (REEMPLAZO de emergencia)", origen="scikit-learn", online=False,
                clases=[str(k) for k in range(10)], H=8, W=8, padding_cnn=1,
                X_tr=X[i_tr], y_tr=y[i_tr], X_va=X[i_va], y_va=y[i_va], X_te=X[i_te], y_te=y[i_te])

def cargar_datos():
    """Devuelve un diccionario con tensores (N, 1, H, W) en [0, 1] y particiones train/val/test."""
    try:
        tr, te = fashion_torchvision(download=True); origen = "torchvision"
    except Exception as e1:
        print(f"[aviso] torchvision no pudo bajar Fashion-MNIST ({type(e1).__name__}); se prueba el espejo de Google.")
        try:
            tr, te = fashion_desde_espejo(); origen = "espejo storage.googleapis.com"
        except Exception as e2:
            print(f"[AVISO] TAMPOCO FUNCIONÓ EL ESPEJO ({type(e2).__name__}). SE USA load_digits (DÍGITOS 8×8) "
                  "COMO REEMPLAZO DE EMERGENCIA: LOS RESULTADOS NO SON LOS DE FASHION-MNIST.")
            return reemplazo_digits()
    X = tr.data.float().div(255).unsqueeze(1); y = tr.targets.clone()   # [0, 255] -> [0, 1] y eje de canal
    g = torch.Generator().manual_seed(SEED)
    idx = torch.randperm(len(X), generator=g)
    nva = len(X) // 10                                                  # 90/10 para validación
    va, trn = idx[:nva], idx[nva:]
    return dict(nombre="Fashion-MNIST", origen=origen, online=True, clases=CLASES_FASHION, H=28, W=28, padding_cnn=0,
                X_tr=X[trn], y_tr=y[trn], X_va=X[va], y_va=y[va],
                X_te=te.data.float().div(255).unsqueeze(1), y_te=te.targets.clone())

DATOS = cargar_datos()
H, W, CLASES = DATOS["H"], DATOS["W"], DATOS["clases"]
X_tr, y_tr, X_va, y_va, X_te, y_te = (DATOS[k] for k in ["X_tr", "y_tr", "X_va", "y_va", "X_te", "y_te"])
BS = 128 if DATOS["online"] else 32   # tamaño de mini-lote por defecto
print(f"Conjunto: {DATOS['nombre']} (vía {DATOS['origen']}) · lote por defecto: {BS}")
print(f"train {tuple(X_tr.shape)} · val {tuple(X_va.shape)} · test {tuple(X_te.shape)}")

# Grilla: una fila por prenda, 8 ejemplos cada una (como la diapositiva 13)
fig, axs = plt.subplots(10, 8, figsize=(8.5, 11))
for c in range(10):
    ejemplos = torch.nonzero(y_tr == c).flatten()[:8]
    for k, ax in enumerate(axs[c]):
        ax.imshow(X_tr[ejemplos[k], 0], cmap="gray_r"); ax.axis("off")
    axs[c, 0].text(-0.15, 0.5, CLASES[c], transform=axs[c, 0].transAxes, ha="right", va="center", fontsize=10)
fig.suptitle(f"{DATOS['nombre']}: 8 ejemplos de cada clase", y=0.995)
plt.tight_layout(); plt.subplots_adjust(left=0.14); plt.show()

# Una foto ampliada: para la red es solo una tabla de números
fig, ax = plt.subplots(figsize=(4.6, 4.8))
ax.imshow(X_tr[0, 0], cmap="gray_r")
ax.set_xticks(np.arange(-0.5, W, 1), minor=True); ax.set_yticks(np.arange(-0.5, H, 1), minor=True)
ax.grid(which="minor", color=GRIS, lw=0.3); ax.grid(which="major", visible=False)
ax.tick_params(which="both", length=0, labelbottom=False, labelleft=False)
ax.set_title(f"{CLASES[int(y_tr[0])]} · {H} × {W} = {H * W} números"); plt.show()
print("Un trozo de esa tabla (0 = blanco, 1 = negro):")
print(np.round(X_tr[0, 0, 10:15, 10:15].numpy(), 2))

### Utilidades (ya implementadas)

In [ ]:
def contar_parametros(modelo):
    """Número total de parámetros entrenables."""
    return sum(p.numel() for p in modelo.parameters() if p.requires_grad)

@torch.no_grad()
def evaluar(modelo, X, y, bs=1024):
    """Devuelve (pérdida, exactitud, predicciones) en modo evaluación."""
    modelo.eval()
    logits = torch.cat([modelo(X[i:i + bs].to(DEVICE)).cpu() for i in range(0, len(X), bs)])
    return F.cross_entropy(logits, y).item(), (logits.argmax(1) == y).float().mean().item(), logits.argmax(1)

def entrenar(modelo, X_tr, y_tr, X_va, y_va, epochs=5, lr=1e-3, batch_size=None,
             augment=None, congelar_bn=False, verbose=True):
    """Ciclo de entrenamiento con Adam. `augment(xb)` se aplica a cada mini-lote de entrenamiento.
    Devuelve el historial: pérdida de entrenamiento, exactitud (train/val) y pérdida de validación por época.
    La exactitud de entrenamiento se mide en modo eval sobre hasta 5000 ejemplos sin aumento."""
    batch_size = batch_size or BS
    modelo.to(DEVICE)
    opt = torch.optim.Adam([p for p in modelo.parameters() if p.requires_grad], lr=lr)
    n, sub = len(X_tr), slice(0, min(len(X_tr), 5000))
    hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for e in range(epochs):
        modelo.train()
        if congelar_bn:   # con backbone congelado, no actualizar las estadísticas de BatchNorm
            for m in modelo.modules():
                if isinstance(m, nn.BatchNorm2d): m.eval()
        perm, total = torch.randperm(n), 0.0
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            xb, yb = X_tr[idx], y_tr[idx]
            if augment is not None: xb = augment(xb)
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = F.cross_entropy(modelo(xb), yb)
            loss.backward(); opt.step()
            total += loss.item() * len(idx)
        _, tr_acc, _ = evaluar(modelo, X_tr[sub], y_tr[sub])
        va_loss, va_acc, _ = evaluar(modelo, X_va, y_va)
        hist["train_loss"].append(total / n); hist["train_acc"].append(tr_acc)
        hist["val_loss"].append(va_loss); hist["val_acc"].append(va_acc)
        if verbose:
            print(f"época {e + 1:>2}/{epochs} · pérdida {total / n:.4f} · acc train {tr_acc:.3f} · "
                  f"pérdida val {va_loss:.4f} · acc val {va_acc:.3f}")
    return hist

def dim_aplanada(features, in_channels, H, W):
    """Número de valores tras la parte convolucional (sirve para dimensionar la primera capa densa)."""
    with torch.no_grad():
        return features(torch.zeros(1, in_channels, H, W)).numel()

def mostrar_grilla(mapas, titulo, ncols=8, cmap=CMAP_SEC, vmin=None, vmax=None, etiquetas=None):
    """Dibuja una lista/arreglo de matrices 2D en una grilla (etiquetas: títulos de cada panel)."""
    n = len(mapas); nrows = math.ceil(n / ncols)
    etiquetas = etiquetas if etiquetas is not None else [f"#{k}" for k in range(n)]
    fig, axs = plt.subplots(nrows, ncols, figsize=(1.6 * ncols, 1.7 * nrows))
    axs = np.atleast_1d(axs).ravel()
    for ax in axs: ax.axis("off")
    for ax, m, et in zip(axs, mapas, etiquetas):
        ax.imshow(m, cmap=cmap, vmin=vmin, vmax=vmax); ax.set_title(et, fontsize=8)
    fig.suptitle(titulo); plt.tight_layout(); plt.show()

def graficar_curvas(historiales, titulo):
    """Compara pérdida de validación y brecha train−val de varios historiales {nombre: hist}."""
    fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
    for nombre, h in historiales.items():
        axs[0].plot(range(1, len(h["val_loss"]) + 1), h["val_loss"], marker="o", ms=3, label=nombre)
        axs[1].plot(range(1, len(h["val_acc"]) + 1), np.array(h["train_acc"]) - np.array(h["val_acc"]),
                    marker="o", ms=3, label=nombre)
    axs[0].set(title="Pérdida de validación", xlabel="época", ylabel="entropía cruzada")
    axs[1].set(title="Brecha exactitud train − val", xlabel="época", ylabel="brecha")
    axs[0].legend(); fig.suptitle(titulo); plt.tight_layout(); plt.show()

RESULTADOS = {}   # aquí se acumulan las exactitudes de prueba por modelo

## Parte A — Línea base: un MLP (15 min)

> **En palabras simples.** El MLP aplana la foto en una lista de 784 números: ya no sabe qué píxel estaba al lado de cuál. Es como desarmar un puzle y echar las piezas a una bolsa. Lo usamos como punto de comparación para la CNN (**diapositiva 13**: la foto como 28 × 28 = 784 números).

Red: `784 → 128 → 64 → 10`, con ReLU entre las capas. Se entrena **5 épocas** con Adam (`lr = 1e-3`) y lotes de 128.

**¿Cómo se cuentan los parámetros?** Cada conexión entre dos neuronas es un número que la red aprende (un *peso*), y cada neurona de llegada tiene además un *sesgo*. Una capa densa que va de `a` a `b` neuronas tiene `a·b + b` parámetros. Sumando las tres capas salen **109.386**: la celda lo comprueba.

In [ ]:
def construir_mlp(n_in, n_classes=10):
    """MLP: Flatten -> Linear(n_in,128) -> ReLU -> Linear(128,64) -> ReLU -> Linear(64,n_classes)."""
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(n_in, 128), nn.ReLU(),
        nn.Linear(128, 64), nn.ReLU(),
        nn.Linear(64, n_classes),
    )

set_seed()
mlp = construir_mlp(H * W)
n_mlp = contar_parametros(mlp)
print("Parámetros del MLP:", n_mlp)
if DATOS["online"]:
    assert n_mlp == 109_386, f"El MLP 784 → 128 → 64 → 10 debería tener 109.386 parámetros; tiene {n_mlp}"
h_mlp = entrenar(mlp, X_tr, y_tr, X_va, y_va, epochs=ep(5))
_, acc_mlp, _ = evaluar(mlp, X_te, y_te)
RESULTADOS["MLP"] = dict(params=n_mlp, acc_test=acc_mlp)
print(f"\nMLP · exactitud en prueba: {acc_mlp:.4f}")

**Pregunta A:** ¿Cuántos parámetros tiene solo la primera capa (784 → 128) y por qué son tantos? ¿Qué porcentaje del total es?

**Respuesta:** (escribe aquí)

## Parte B — La primera CNN (25 min)

> **En palabras simples.** En vez de aplanar la foto, la CNN la recorre con lupas pequeñas de 3 × 3: los *filtros*. Cada lupa busca un patrón (por ejemplo, un borde) en todas partes de la foto, y usa los mismos 9 números en cada lugar. Por eso necesita muchos menos parámetros. La arquitectura es la de la **diapositiva 14**.

`Entrada 1×28×28 → Conv(16, 3×3) + ReLU → MaxPool(2) → Conv(32, 3×3) + ReLU → MaxPool(2) → Flatten → Dense(64) + ReLU → Dense(10)`

Cómo cambia el tamaño, paso a paso:

| Paso | Qué pasa | Forma |
|---|---|---|
| Entrada | una foto en gris | 1 × 28 × 28 |
| Conv(16, 3×3) + ReLU | 16 filtros; la lupa de 3×3 no cabe en el borde, así que se pierde 1 píxel por lado | 16 × 26 × 26 |
| MaxPool(2) | de cada cuadrito de 2×2 queda el valor más alto: la mitad de alto y de ancho | 16 × 13 × 13 |
| Conv(32, 3×3) + ReLU | 32 filtros que miran los 16 mapas anteriores a la vez | 32 × 11 × 11 |
| MaxPool(2) | otra vez la mitad (11 / 2 → 5; el píxel que sobra se descarta) | 32 × 5 × 5 |
| Flatten | todo en una fila: 32 · 5 · 5 | 800 |
| Dense(64) + ReLU → Dense(10) | igual que en el MLP | 64 → 10 |

Para quien quiera la regla general (después de la intuición): tamaño de salida de una convolución = `⌊(n + 2p − k) / s⌋ + 1`, con n = tamaño de entrada, k = tamaño del filtro, p = relleno y s = paso. Parámetros de una capa convolucional = `(k · k · canales de entrada + 1) · número de filtros`.

En el código, la parte convolucional va en `self.features` (termina en `MaxPool2d`) y la parte densa en `self.classifier` (empieza en `nn.Flatten()`).

**TODO.** Completa las capas de `CNNSimple.__init__`. El autochequeo revisa las formas de la tabla y los **56.714** parámetros.

In [ ]:
class CNNSimple(nn.Module):
    """Conv(16,3x3)+ReLU -> MaxPool(2) -> Conv(32,3x3)+ReLU -> MaxPool(2) -> Flatten -> Dense(64)+ReLU -> Dense(n_classes)."""
    def __init__(self, in_channels=1, H=28, W=28, n_classes=10, padding=0):
        super().__init__()
        # TODO 1: self.features = nn.Sequential(Conv2d -> ReLU -> MaxPool2d -> Conv2d -> ReLU -> MaxPool2d)
        #         pistas: nn.Conv2d(canales_entrada, filtros, kernel_size=3, padding=padding); 16 y luego 32 filtros;
        #                 nn.MaxPool2d(2). La primera Conv2d recibe `in_channels` canales.
        self.features = ...
        n_flat = dim_aplanada(self.features, in_channels, H, W)
        # TODO 2: self.classifier = nn.Sequential(Flatten -> Linear(n_flat, 64) -> ReLU -> Linear(64, n_classes))
        self.classifier = ...

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
def trazar_formas(modelo, x):
    """Imprime y devuelve la forma (sin el eje del lote) tras cada capa."""
    formas = []
    print(f"{'entrada':<12}{tuple(x.shape)}")
    for capa in list(modelo.features) + list(modelo.classifier):
        x = capa(x)
        formas.append((capa.__class__.__name__, tuple(x.shape[1:])))
        print(f"{capa.__class__.__name__:<12}{tuple(x.shape)}")
    return formas

# Autochequeo: arquitectura de la diapositiva 14 sobre una entrada 1×28×28
try:
    modelo_ref = CNNSimple(in_channels=1, H=28, W=28, n_classes=10, padding=0)
except TypeError as e:
    raise AssertionError("CNNSimple todavía no está completa: revisa los TODO 1 y 2 "
                         "(self.features y self.classifier deben ser nn.Sequential con sus capas).") from e
formas = trazar_formas(modelo_ref, torch.zeros(1, 1, 28, 28))
esperadas = [("Conv2d", (16, 26, 26)), ("ReLU", (16, 26, 26)), ("MaxPool2d", (16, 13, 13)),
             ("Conv2d", (32, 11, 11)), ("ReLU", (32, 11, 11)), ("MaxPool2d", (32, 5, 5)),
             ("Flatten", (800,)), ("Linear", (64,)), ("ReLU", (64,)), ("Linear", (10,))]
assert len(formas) == len(esperadas), (f"La red debería tener {len(esperadas)} capas en total "
                                       f"(6 en features y 4 en classifier); tiene {len(formas)}")
for k, ((nombre, forma), (nombre_ok, forma_ok)) in enumerate(zip(formas, esperadas), start=1):
    assert nombre == nombre_ok, f"Revisa la capa {k}: debería ser {nombre_ok} y es {nombre}"
    assert forma == forma_ok, f"Revisa la forma tras la capa {k} ({nombre}): debería ser {forma_ok} y salió {forma}"
n_ref = contar_parametros(modelo_ref)
assert n_ref == 56_714, f"Revisa las capas: la CNN debería tener 56.714 parámetros y tiene {n_ref:,}".replace(",", ".")
print("\nOK: las formas son las de la diapositiva 14 (16×26×26 → 16×13×13 → 32×11×11 → 32×5×5 → 800) "
      "y hay 56.714 parámetros.")

# Desglose por capa (para comparar con el cálculo a mano)
for nombre, p in modelo_ref.named_parameters():
    print(f"{nombre:<22}{str(tuple(p.shape)):<18}{p.numel():>8}")

Ahora se entrena la CNN **5 épocas** (Adam, `lr = 1e-3`, lotes de 128) y se compara con el MLP.

In [ ]:
set_seed()
cnn = CNNSimple(in_channels=1, H=H, W=W, n_classes=10, padding=DATOS["padding_cnn"])
print("Parámetros de la CNN:", contar_parametros(cnn))
h_cnn = entrenar(cnn, X_tr, y_tr, X_va, y_va, epochs=ep(5))
_, acc_cnn, pred_cnn = evaluar(cnn, X_te, y_te)
RESULTADOS["CNN simple"] = dict(params=contar_parametros(cnn), acc_test=acc_cnn)

comparacion = pd.DataFrame(RESULTADOS).T.rename(columns={"params": "parámetros", "acc_test": "exactitud en prueba"})
comparacion["parámetros"] = comparacion["parámetros"].astype(int)
display(comparacion.style.format({"exactitud en prueba": "{:.4f}"}))

### Matriz de confusión y F1 por clase

La **matriz de confusión** cuenta, para cada prenda real (fila), qué respondió la red (columna). La diagonal son los aciertos; todo lo que queda fuera de ella son confusiones. El **F1** resume en un número entre 0 y 1 qué tan bien le va a la red con cada prenda, y el **F1 macro** es el promedio de las 10 prendas (Clase 9).

**TODO.** Calcula la matriz de confusión de la CNN en prueba, dibújala con los nombres de las prendas y calcula el F1 por clase y el F1 macro.

In [ ]:
# TODO 1: matriz de confusión con confusion_matrix(y_te, pred_cnn, labels=range(10))
cm = ...
# TODO 2: dibújala con ConfusionMatrixDisplay(cm, display_labels=CLASES).plot(ax=ax, cmap=CMAP_SEC,
#         colorbar=False, xticks_rotation=45)
fig, ax = plt.subplots(figsize=(7, 6.5))
...
ax.set(title="Matriz de confusión · CNN simple (prueba)", xlabel="predicción", ylabel="prenda real"); ax.grid(False)
plt.show()

# TODO 3: F1 por clase con f1_score(y_te, pred_cnn, average=None, labels=range(10)) y F1 macro con average="macro"
f1_clase = ...
f1_macro = ...

In [ ]:
# Autochequeo
assert isinstance(cm, np.ndarray) and cm.shape == (10, 10), \
    "Revisa la matriz: debería ser un arreglo de 10 × 10 (una fila y una columna por prenda)"
assert cm.sum() == len(y_te), f"La matriz debería sumar {len(y_te)} (una cuenta por cada foto de prueba); suma {cm.sum()}"
assert (cm.sum(axis=1) == np.bincount(y_te.numpy(), minlength=10)).all(), \
    "Las filas deberían ser la prenda real: pasa primero y_te y después pred_cnn"
assert np.ndim(f1_clase) == 1 and len(f1_clase) == 10, "f1_clase debería tener 10 valores (uno por prenda): usa average=None"
assert np.isclose(f1_macro, np.mean(f1_clase)), "El F1 macro debería ser el promedio de los 10 F1 por clase: usa average=\"macro\""
print("OK: matriz de confusión y F1 correctos.")

print(f"F1 macro: {f1_macro:.4f}")
display(pd.DataFrame({"prenda": CLASES, "F1": f1_clase}).sort_values("F1").style.format({"F1": "{:.3f}"}))

def top_confusiones(cm, clases, k=5):
    """Pares (real -> predicho) más frecuentes fuera de la diagonal."""
    fuera = cm.copy(); np.fill_diagonal(fuera, 0)
    orden = np.dstack(np.unravel_index(np.argsort(-fuera, axis=None), fuera.shape))[0][:k]
    return [(clases[i], clases[j], int(fuera[i, j])) for i, j in orden]

print("Confusiones más frecuentes (real → predicción):")
for real, predicho, n in top_confusiones(cm, CLASES):
    print(f"  {real} → {predicho}: {n} casos")

**Pregunta B:** ¿Qué prendas se confunden más y por qué tiene sentido? ¿Qué ganó la CNN frente al MLP (exactitud y parámetros)?

**Respuesta:** (escribe aquí)

## Parte C — Mirar por dentro (20 min)

> **En palabras simples.** Una CNN aprende sus propios filtros. Aquí miramos qué aprendió: los 16 filtros de la primera capa y lo que «ve» cada uno en una foto. Son los mismos tipos de filtros que viste en la clase (**diapositiva 8**).

- **C1.** Los 16 filtros 3×3 de la primera capa. Azul = peso negativo, blanco = cero, naranja = positivo (los colores de las diapositivas 7 y 8).
- **C2.** Los mapas de características de un botín de prueba: qué partes de la foto «encienden» cada filtro.
- **C3.** Qué hace la ReLU con esos mapas (diapositiva 6).
- **C4.** Una convolución hecha a mano con NumPy.

**TODO.** Extrae los pesos de la primera capa (C1), calcula los mapas (C2) y completa `conv2d_np` (C4). Cada una tiene su autochequeo.

### C1. Los filtros aprendidos

In [ ]:
# TODO: extrae los pesos de cnn.features[0] como arreglo NumPy de forma (16, 3, 3)
#       pistas: cnn.features[0].weight tiene forma (16, 1, 3, 3); usa .detach().cpu().numpy() y quita el eje del canal
filtros = ...

In [ ]:
# Autochequeo
assert isinstance(filtros, np.ndarray), "filtros debería ser un arreglo de NumPy: usa .detach().cpu().numpy()"
assert filtros.shape == (16, 3, 3), f"Revisa la forma: debería ser (16, 3, 3) y salió {filtros.shape} (¿quitaste el eje del canal con [:, 0]?)"
assert np.allclose(filtros, cnn.features[0].weight.detach().cpu().numpy()[:, 0]), \
    "Los valores no coinciden con los pesos de la primera capa (cnn.features[0])"
print("OK: 16 filtros de 3×3 de la primera capa.")

In [ ]:
vmax = np.abs(filtros).max()                                         # escala simétrica: el 0 queda en blanco
mostrar_grilla(filtros, "Filtros 3×3 de la primera capa (azul = negativo · blanco = 0 · naranja = positivo)",
               cmap=CMAP_DIV, vmin=-vmax, vmax=vmax)

### C2. Mapas de características de un botín

Se pasa una foto de prueba (`X_te[0]`, un botín) por la red y se guarda lo que sale de cada capa. Cada filtro produce un **mapa**: una imagen que se ilumina donde el filtro encontró su patrón.

- `a1`: salida de Conv1 + ReLU → 16 mapas de 26×26.
- `a2`: salida de MaxPool → Conv2 + ReLU → 32 mapas de 11×11 (se muestran los 16 más activos).

In [ ]:
i_img = 0
img = X_te[i_img:i_img + 1].to(DEVICE)
print(cnn.features)   # mira el orden de las capas: features[0], features[1], …
with torch.no_grad():
    # TODO: a1 = salida de Conv1 + ReLU (pasa img por cnn.features[0] y luego por cnn.features[1])
    #       a2 = salida de MaxPool + Conv2 + ReLU (pasa a1 por cnn.features[2], [3] y [4])
    a1 = ...
    a2 = ...

In [ ]:
# Autochequeo
PAD = DATOS["padding_cnn"]
forma_a1 = (1, 16, H + 2 * PAD - 2, W + 2 * PAD - 2)
forma_a2 = (1, 32, (forma_a1[2] // 2) + 2 * PAD - 2, (forma_a1[3] // 2) + 2 * PAD - 2)
assert torch.is_tensor(a1) and tuple(a1.shape) == forma_a1, \
    f"Revisa la forma de a1: debería ser {forma_a1} (16 mapas de {forma_a1[2]}×{forma_a1[3]}) y salió {tuple(getattr(a1, 'shape', ()))}"
assert a1.min() >= 0, "a1 tiene valores negativos: ¿te faltó pasar por la ReLU (cnn.features[1])?"
assert torch.is_tensor(a2) and tuple(a2.shape) == forma_a2, \
    f"Revisa la forma de a2: debería ser {forma_a2} y salió {tuple(getattr(a2, 'shape', ()))} (¿pasaste por el MaxPool antes de Conv2?)"
with torch.no_grad():
    assert torch.allclose(a2, cnn.features[:5](img)), "a2 no coincide con la salida de las capas 0 a 4 de cnn.features"
print(f"OK: a1 {tuple(a1.shape)} · a2 {tuple(a2.shape)}")

In [ ]:
plt.figure(figsize=(2.4, 2.6)); plt.imshow(img[0, 0].cpu(), cmap="gray_r"); plt.axis("off")
plt.title(f"Entrada: {CLASES[int(y_te[i_img])]}"); plt.show()
print("Forma tras Conv1 + ReLU:", tuple(a1.shape), "· tras Conv2 + ReLU:", tuple(a2.shape))

mostrar_grilla(a1[0].cpu().numpy(), "Mapas de Conv1 + ReLU (16 filtros, 26×26)")
activos = a2[0].mean(dim=(1, 2)).argsort(descending=True)[:16].cpu()      # los 16 mapas con más activación
mostrar_grilla(a2[0, activos].cpu().numpy(), "Mapas de Conv2 + ReLU (los 16 más activos de 32, 11×11)",
               etiquetas=[f"#{int(k)}" for k in activos])

apagados = (a1[0].amax(dim=(1, 2)) == 0).sum().item()
print(f"Filtros de Conv1 que no se activan en ninguna parte de esta foto: {apagados} de 16")

### C3. Qué hace la ReLU

La ReLU cambia cada número negativo por 0 y deja igual los positivos. Antes de la ReLU, el mapa de un filtro tiene valores positivos (naranja: «aquí está mi patrón») y negativos (azul: «aquí está lo contrario»). Después, solo queda lo naranja: **la ReLU deja solo los lugares donde el filtro encontró su patrón**. Es el mismo mensaje de la **diapositiva 6**.

In [ ]:
with torch.no_grad():
    z1 = cnn.features[0](img)[0].cpu()        # Conv1 ANTES de la ReLU: tiene positivos y negativos
r1 = torch.relu(z1)                           # DESPUÉS de la ReLU: solo positivos
sin_sesgo = z1 - cnn.features[0].bias.detach().cpu()[:, None, None]       # sin el valor constante del fondo
equilibrio = torch.minimum(sin_sesgo.clamp(min=0).sum(dim=(1, 2)), (-sin_sesgo).clamp(min=0).sum(dim=(1, 2)))
dos = equilibrio.argsort(descending=True)[:2]   # 2 detectores de bordes: mapas con mucho positivo Y mucho negativo

fig, axs = plt.subplots(2, 2, figsize=(6.4, 6.6))
for fila, k in enumerate(dos.tolist()):
    lim = z1[k].abs().max().item()
    axs[fila, 0].imshow(z1[k], cmap=CMAP_DIV, vmin=-lim, vmax=lim)
    axs[fila, 0].set_title(f"Filtro #{k} · antes de la ReLU", fontsize=9)
    axs[fila, 1].imshow(r1[k], cmap=CMAP_DIV, vmin=-lim, vmax=lim)
    axs[fila, 1].set_title(f"Filtro #{k} · después de la ReLU", fontsize=9)
    print(f"Filtro #{k}: {(z1[k] < 0).float().mean():.0%} de los píxeles eran negativos y la ReLU los dejó en 0")
for a in axs.ravel(): a.axis("off")
fig.suptitle("La ReLU deja solo los lugares donde el filtro encontró su patrón"); plt.tight_layout(); plt.show()

### C4. Una convolución a mano

Cómo se calcula un número del mapa: se pone la lupa de 3×3 sobre un rincón de la foto, se multiplica cada píxel por el número del filtro que le queda encima y se suman los 9 resultados. Luego se corre la lupa un paso y se repite, fila por fila. Eso es todo lo que hace una capa convolucional (sin la ReLU).

Se usa el filtro de bordes verticales de la **diapositiva 6**: `[1, 0, −1]` repetido en 3 filas. Responde fuerte donde hay un cambio de claro a oscuro de izquierda a derecha.

Con **relleno** (`padding`) se agrega un marco de ceros alrededor de la foto, y con **paso** (`stride`) la lupa salta más de un píxel a la vez. La regla del tamaño de salida es `⌊(n + 2p − k) / s⌋ + 1`.

**TODO.** Completa `conv2d_np`.

In [ ]:
def conv2d_np(img, kernel, stride=1, padding=0):
    """Correlación cruzada 2D (lo que las librerías llaman "convolución") de una imagen 2D con un filtro 2D."""
    img = np.pad(img, padding)
    kh, kw = kernel.shape
    # TODO 1: tamaño de la salida con la fórmula ⌊(n + 2p − k)/s⌋ + 1
    #         (ojo: img ya incluye el relleno, así que aquí basta (img.shape[0] - kh) // stride + 1)
    h_out = ...
    w_out = ...
    salida = np.zeros((h_out, w_out))
    # TODO 2: recorre las posiciones (i, j); toma el parche
    #         img[i*stride : i*stride + kh, j*stride : j*stride + kw],
    #         multiplícalo elemento a elemento por el filtro y guarda la suma en salida[i, j]
    ...
    return salida

In [ ]:
# Autochequeo con un ejemplo que se puede hacer a mano: imagen 4×4 con los números 0..15 y un filtro de unos
try:
    prueba = conv2d_np(np.arange(16.0).reshape(4, 4), np.ones((3, 3)))
except TypeError as e:
    raise AssertionError("conv2d_np todavía no está completa: revisa el TODO 1 (h_out y w_out deben ser números enteros)") from e
assert isinstance(prueba, np.ndarray) and prueba.shape == (2, 2), \
    f"Revisa la forma: con una imagen 4×4 y un filtro 3×3 la salida debería ser (2, 2) y salió {getattr(prueba, 'shape', None)}"
assert np.allclose(prueba, [[45, 54], [81, 90]]), (
    "Revisa el TODO 2: cada casilla es la suma de (parche × filtro). Arriba a la izquierda debería dar "
    f"0+1+2+4+5+6+8+9+10 = 45; obtuve\n{prueba}")
print("OK: conv2d_np da [[45, 54], [81, 90]], igual que la cuenta a mano.")

In [ ]:
borde_vertical = np.array([[1, 0, -1],
                           [1, 0, -1],
                           [1, 0, -1]], dtype=np.float32)
i_prenda = int(torch.nonzero(y_te == 1)[0])     # la primera prenda de clase 1 en prueba (un pantalón)
imagen = X_te[i_prenda, 0].numpy()

salida_np = conv2d_np(imagen, borde_vertical, stride=1, padding=0)

# Comparación con PyTorch: F.conv2d y una capa nn.Conv2d con ese filtro fijo
salida_f = F.conv2d(torch.tensor(imagen)[None, None], torch.tensor(borde_vertical)[None, None])[0, 0].numpy()
capa = nn.Conv2d(1, 1, kernel_size=3, bias=False)
with torch.no_grad(): capa.weight.copy_(torch.tensor(borde_vertical)[None, None])
salida_capa = capa(torch.tensor(imagen)[None, None])[0, 0].detach().numpy()

assert salida_np.shape == (H - 2, W - 2), f"Revisa la forma: debería ser {(H - 2, W - 2)} y salió {salida_np.shape}"
assert np.allclose(salida_np, salida_f, atol=1e-5), "No coincide con F.conv2d"
assert np.allclose(salida_np, salida_capa, atol=1e-5), "No coincide con nn.Conv2d"
print(f"OK: la convolución a mano coincide con F.conv2d y con nn.Conv2d (forma de salida {salida_np.shape}).")

# Con relleno y paso: verificar la regla del tamaño
for p, s in [(1, 1), (0, 2), (1, 2)]:
    esperado = (H + 2 * p - 3) // s + 1
    obtenido = conv2d_np(imagen, borde_vertical, stride=s, padding=p).shape
    assert obtenido == (esperado, esperado), f"Con p={p}, s={s} debería salir {(esperado, esperado)} y salió {obtenido}"
    print(f"  p={p}, s={s}: ⌊({H} + 2·{p} − 3)/{s}⌋ + 1 = {esperado}  ✓")
print("OK: la regla ⌊(n + 2p − k)/s⌋ + 1 se cumple para (p, s) ∈ {(1, 1), (0, 2), (1, 2)}.")

lim = np.abs(salida_np).max()
fig, axs = plt.subplots(1, 3, figsize=(9.5, 3.4))
axs[0].imshow(imagen, cmap="gray_r"); axs[0].set_title(f"Foto: {CLASES[int(y_te[i_prenda])]}")
axs[1].imshow(borde_vertical, cmap=CMAP_DIV, vmin=-1, vmax=1); axs[1].set_title("Filtro de bordes verticales")
axs[2].imshow(salida_np, cmap=CMAP_DIV, vmin=-lim, vmax=lim); axs[2].set_title("Salida (NumPy)")
for a in axs: a.axis("off")
plt.tight_layout(); plt.show()

**Pregunta C:** ¿Qué detecta cada filtro de la primera capa? ¿En qué se diferencian los mapas de la capa 1 y los de la capa 2?

**Respuesta:** (escribe aquí)

## Parte D — Regularizar y aumentar datos (20 min)

> **En palabras simples.** Una red puede «memorizarse» las fotos de entrenamiento y después fallar con fotos nuevas. Aquí se prueban dos remedios y se mide la **brecha**: cuánto mejor le va en entrenamiento que en validación. Mientras más pequeña la brecha, mejor generaliza. Es lo de la **diapositiva 16**.

Tres configuraciones, **10 épocas** cada una y la misma semilla:

1. **Base:** la `CNNSimple` de la Parte B.
2. **BN + Dropout:** BatchNorm después de cada convolución (mantiene los números en una escala pareja) y Dropout de 0,3 en la parte densa (en cada paso apaga al azar el 30 % de las neuronas, para que la red no dependa de unas pocas).
3. **BN + Dropout + aumento:** además, en cada lote la red ve versiones levemente cambiadas de las fotos.

### El aumento debe respetar la etiqueta

Aumentar datos es mostrarle a la red la misma foto con cambios pequeños, para que aprenda lo importante y no los detalles. La regla de oro de la diapositiva 16: **el cambio no puede cambiar la respuesta**. Una zapatilla volteada como en un espejo sigue siendo una zapatilla, y correrla 2 píxeles tampoco cambia nada. Lo que no se hace es ponerla de cabeza: una prenda de cabeza no aparece en las fotos reales, y en otros problemas eso cambia el significado (un 6 girado se vuelve 9).

Aquí se usan dos cambios:

- **Volteo horizontal**, la mitad de las veces.
- **Recorte aleatorio**: se agrega un marco de 2 píxeles y se vuelve a recortar a 28×28 en una posición al azar. En la práctica, la foto se corre hasta 2 píxeles.

> **Nota de rendimiento.** Aplicar los cambios foto por foto es lento con 54.000 imágenes. La función `aumentar` los aplica a todo el lote de una vez y da el mismo resultado; la celda lo comprueba y mide la diferencia de tiempo.

In [ ]:
class CNNReg(nn.Module):
    """CNN con BatchNorm2d tras cada convolución y Dropout en la parte densa."""
    def __init__(self, in_channels=1, H=28, W=28, n_classes=10, padding=0, bn=True, p_dropout=0.3):
        super().__init__()
        def bloque(c_in, c_out):
            capas = [nn.Conv2d(c_in, c_out, kernel_size=3, padding=padding)]
            if bn: capas.append(nn.BatchNorm2d(c_out))
            return capas + [nn.ReLU(), nn.MaxPool2d(2)]
        self.features = nn.Sequential(*bloque(in_channels, 16), *bloque(16, 32))
        n_flat = dim_aplanada(self.features, in_channels, H, W)
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(p_dropout), nn.Linear(n_flat, 64), nn.ReLU(),
            nn.Dropout(p_dropout), nn.Linear(64, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
RELLENO, VOLTEAR = (2, True) if DATOS["online"] else (1, False)   # en el reemplazo de emergencia no se voltea

def aumentar(xb, pad=RELLENO, voltear=VOLTEAR):
    """Aumento POR LOTE: volteo horizontal al azar (50 %) y recorte aleatorio con relleno `pad`.
    Equivale a T.Compose([T.RandomHorizontalFlip(), T.RandomCrop(H, padding=pad)]) aplicado foto por foto."""
    n, _, h, w = xb.shape
    if voltear:
        al_espejo = torch.rand(n) < 0.5                                      # máscara: qué fotos se voltean
        xb = torch.where(al_espejo[:, None, None, None], xb.flip(-1), xb)
    xp = F.pad(xb, (pad, pad, pad, pad))                                     # marco de ceros
    dy = torch.randint(0, 2 * pad + 1, (n,)); dx = torch.randint(0, 2 * pad + 1, (n,))   # desplazamiento por foto
    filas = dy[:, None, None] + torch.arange(h)[None, :, None]               # n × h × 1
    cols = dx[:, None, None] + torch.arange(w)[None, None, :]                # n × 1 × w
    return xp[torch.arange(n)[:, None, None], :, filas, cols].permute(0, 3, 1, 2).contiguous()

# Comprobación: cada foto aumentada es exactamente uno de los resultados posibles de la versión foto por foto
set_seed()
muestra = X_tr[:64]
aum = aumentar(muestra)
assert aum.shape == muestra.shape
for x, xa in zip(muestra, aum):
    candidatos = [TF.crop(TF.pad(v, RELLENO), top, left, H, W)
                  for v in ([x, TF.hflip(x)] if VOLTEAR else [x])
                  for top in range(2 * RELLENO + 1) for left in range(2 * RELLENO + 1)]
    assert any(torch.equal(xa, c) for c in candidatos), "aumentar no equivale a RandomHorizontalFlip + RandomCrop"
print("OK: aumentar (por lote) equivale a volteo + recorte foto por foto.")

TF_AUG = T.Compose([T.RandomHorizontalFlip(p=0.5 if VOLTEAR else 0.0), T.RandomCrop(H, padding=RELLENO)])
t = time.time(); torch.stack([TF_AUG(x) for x in X_tr[:5000]]); t_foto = time.time() - t
t = time.time(); aumentar(X_tr[:5000]); t_lote = time.time() - t
print(f"5.000 fotos: foto por foto {t_foto:.2f} s · por lote {t_lote:.3f} s (≈{t_foto / t_lote:.0f} veces más rápido)")

# Vista previa: una prenda y cuatro versiones aumentadas
set_seed()
fig, axs = plt.subplots(1, 5, figsize=(10, 2.4))
axs[0].imshow(X_tr[0, 0], cmap="gray_r"); axs[0].set_title(f"original ({CLASES[int(y_tr[0])]})", fontsize=9)
versiones = aumentar(X_tr[0:1].repeat(4, 1, 1, 1))
for k in range(4):
    axs[k + 1].imshow(versiones[k, 0], cmap="gray_r"); axs[k + 1].set_title(f"aumento {k + 1}", fontsize=9)
for a in axs: a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
PAD = DATOS["padding_cnn"]
configs = {
    "Base":                   (lambda: CNNSimple(1, H, W, 10, PAD), None),
    "BN + Dropout":           (lambda: CNNReg(1, H, W, 10, PAD, bn=True, p_dropout=0.3), None),
    "BN + Dropout + aumento": (lambda: CNNReg(1, H, W, 10, PAD, bn=True, p_dropout=0.3), aumentar),
}

historiales, filas = {}, []
for nombre, (fabrica, aug) in configs.items():
    t = time.time()
    set_seed()
    m = fabrica()
    h = entrenar(m, X_tr, y_tr, X_va, y_va, epochs=ep(10), augment=aug, verbose=False)
    _, acc_test, pred = evaluar(m, X_te, y_te)
    historiales[nombre] = h
    filas.append({"configuración": nombre, "parámetros": contar_parametros(m),
                  "acc train": h["train_acc"][-1], "acc val": h["val_acc"][-1],
                  "brecha (train − val)": h["train_acc"][-1] - h["val_acc"][-1],
                  "acc prueba": acc_test, "F1 macro prueba": f1_score(y_te, pred, average="macro")})
    print(f"{nombre:<24} acc train {h['train_acc'][-1]:.3f} · acc val {h['val_acc'][-1]:.3f} · {time.time() - t:.0f} s")

tabla_d = pd.DataFrame(filas).set_index("configuración")
display(tabla_d.style.format({c: "{:.4f}" for c in tabla_d.columns if c != "parámetros"}))
graficar_curvas(historiales, "Efecto de BatchNorm, Dropout y aumento de datos")

**Pregunta D:** ¿Qué técnica mejoró más la generalización? ¿Se puede concluir algo firme de diferencias pequeñas con una sola semilla?

**Respuesta:** (escribe aquí)

## Parte E — Transferencia de aprendizaje (opcional, 25 min, GPU recomendada)

> **En palabras simples.** En vez de partir de cero, se usa una red que ya aprendió a «ver» con más de un millón de fotos (ResNet-18, entrenada con ImageNet). Sus capas quedan **congeladas**: no se mueven. Solo se le cambia la última capa (la «cabeza») y se entrena esa. Es la animación de la **diapositiva 19**.

- **Datos:** CIFAR-10 (fotos a color de 32×32), solo 3 clases: avión, gato y barco. 500 fotos de entrenamiento y 200 de prueba por clase, agrandadas a 64×64 y ajustadas con las medias de ImageNet.
- **Modelo:** `resnet18(weights="DEFAULT")`, todo congelado y una `fc` nueva de 512 → 3, o sea 512 · 3 + 3 = **1.539** parámetros entrenables. Se entrena 3 épocas.
- **Comparación:** la `CNNSimple` entrenada desde cero 10 épocas con los mismos datos.

> Requiere descargar CIFAR-10 (unos 170 MB) y los pesos de ResNet-18 (unos 45 MB). En CPU puede tardar unos minutos.

In [ ]:
CLASES_E = {0: "avión", 3: "gato", 8: "barco"}     # clases de CIFAR-10 elegidas
MEDIA_IMNET = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD_IMNET = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def cargar_cifar_subset(n_train=500, n_test=200, tam=64):
    """Subconjunto de CIFAR-10 (3 clases) redimensionado y normalizado. Devuelve None si no hay descarga."""
    try:
        from torchvision import datasets
        tr = datasets.CIFAR10("data", train=True, download=True)
        te = datasets.CIFAR10("data", train=False, download=True)
    except Exception as e:
        print(f"[AVISO] CIFAR-10 no disponible ({type(e).__name__}): requiere descarga.")
        return None
    def preparar(imgs, etiquetas, n_por_clase):
        etiquetas = np.array(etiquetas); xs, ys = [], []
        for nueva, original in enumerate(CLASES_E):
            idx = np.where(etiquetas == original)[0][:n_por_clase]
            xs.append(torch.tensor(imgs[idx]).permute(0, 3, 1, 2).float() / 255); ys += [nueva] * len(idx)
        x = F.interpolate(torch.cat(xs), size=(tam, tam), mode="bilinear", align_corners=False)
        return (x - MEDIA_IMNET) / STD_IMNET, torch.tensor(ys)
    X1, y1 = preparar(tr.data, tr.targets, n_train)
    X2, y2 = preparar(te.data, te.targets, n_test)
    return dict(X_tr=X1, y_tr=y1, X_te=X2, y_te=y2)

DATOS_E = cargar_cifar_subset()
if DATOS_E is not None:
    print({k: tuple(v.shape) for k, v in DATOS_E.items()})
    fig, axs = plt.subplots(1, 9, figsize=(12, 1.8))
    for ax, i in zip(axs, [0, 1, 2, 500, 501, 502, 1000, 1001, 1002]):
        ax.imshow((DATOS_E["X_tr"][i:i + 1] * STD_IMNET + MEDIA_IMNET)[0].permute(1, 2, 0).clamp(0, 1))
        ax.set_title(list(CLASES_E.values())[int(DATOS_E["y_tr"][i])], fontsize=9); ax.axis("off")
    plt.tight_layout(); plt.show()

In [ ]:
from torchvision import models

def armar_resnet18(n_classes, preentrenada=True):
    """ResNet-18 con el backbone congelado y una cabeza `fc` nueva. Devuelve (modelo, ¿pesos de ImageNet cargados?)."""
    pesos_ok, m = False, None
    if preentrenada:
        try:
            m = models.resnet18(weights="DEFAULT"); pesos_ok = True
        except Exception as e:
            print(f"[AVISO] Sin pesos de ImageNet ({type(e).__name__}): se usan pesos ALEATORIOS (solo validación de lógica).")
    if m is None:
        m = models.resnet18(weights=None)
    for p in m.parameters():
        p.requires_grad = False                         # congelar el backbone
    m.fc = nn.Linear(m.fc.in_features, n_classes)       # cabeza nueva: requires_grad=True por defecto
    return m, pesos_ok

set_seed()
resnet, PESOS_IMAGENET = armar_resnet18(n_classes=len(CLASES_E))
n_entrenables = contar_parametros(resnet)
print(f"Pesos de ImageNet cargados: {PESOS_IMAGENET}")
print(f"Parámetros entrenables: {n_entrenables:,} de {sum(p.numel() for p in resnet.parameters()):,}".replace(",", "."))
assert n_entrenables == 1_539, f"Solo la cabeza fc debería ser entrenable: 512·3 + 3 = 1.539 (salen {n_entrenables})"

# Validación de la lógica con un lote sintético: el gradiente solo debe llegar a la cabeza nueva
resnet.train()
x_falso = torch.randn(8, 3, 64, 64); y_falso = torch.randint(0, len(CLASES_E), (8,))
F.cross_entropy(resnet(x_falso), y_falso).backward()
assert resnet.conv1.weight.grad is None, "El backbone debería estar congelado"
assert resnet.fc.weight.grad is not None, "La cabeza nueva debería recibir gradiente"
print("OK: el backbone está congelado y solo la cabeza fc recibe gradiente.")

In [ ]:
# Entrenamiento de la cabeza (3 épocas) y comparación con la CNN de la Parte B (mismos datos)
if DATOS_E is None:
    print("Requiere descarga (CIFAR-10 y pesos de ImageNet); se omite el entrenamiento.")
else:
    t = time.time()
    set_seed()
    resnet, _ = armar_resnet18(n_classes=len(CLASES_E))
    h_res = entrenar(resnet, DATOS_E["X_tr"], DATOS_E["y_tr"], DATOS_E["X_te"], DATOS_E["y_te"],
                     epochs=3, lr=1e-3, batch_size=64, congelar_bn=True)
    _, acc_res, _ = evaluar(resnet, DATOS_E["X_te"], DATOS_E["y_te"])
    print(f"ResNet-18: {time.time() - t:.0f} s\n")

    t = time.time()
    set_seed()
    cnn_e = CNNSimple(in_channels=3, H=64, W=64, n_classes=len(CLASES_E), padding=0)
    h_cnn_e = entrenar(cnn_e, DATOS_E["X_tr"], DATOS_E["y_tr"], DATOS_E["X_te"], DATOS_E["y_te"],
                       epochs=10, lr=1e-3, batch_size=64)
    _, acc_cnn_e, _ = evaluar(cnn_e, DATOS_E["X_te"], DATOS_E["y_te"])
    print(f"CNN desde cero: {time.time() - t:.0f} s")
    display(pd.DataFrame({"modelo": ["ResNet-18 preentrenada (solo cabeza nueva, 3 épocas)", "CNN simple desde cero (10 épocas)"],
                          "parámetros entrenables": [contar_parametros(resnet), contar_parametros(cnn_e)],
                          "exactitud en prueba": [acc_res, acc_cnn_e]}).set_index("modelo")
            .style.format({"exactitud en prueba": "{:.4f}"}))

**Pregunta E:** ¿Qué ventaja da partir de una red preentrenada cuando hay pocos datos? ¿Qué cambiaría al descongelar el backbone?

**Respuesta:** (escribe aquí)

In [ ]:
print(f"Tiempo total del notebook: {(time.time() - T0) / 60:.1f} min (incluye descargas si las hubo)")

## Entregable

Notebook `Lab11_FashionMNIST_CIFAR10_estudiante.ipynb` con las partes **A–D** ejecutadas (la **E** es opcional) y estas tres respuestas breves:

1. ¿Qué ganó la CNN frente al MLP y por qué?
2. ¿Qué hizo cada filtro?
3. ¿Qué técnica de la Parte D mejoró más la generalización?

Entrega por AULA-USM.

**1. ¿Qué ganó la CNN frente al MLP y por qué?**

(escribe aquí)

**2. ¿Qué hizo cada filtro?**

(escribe aquí)

**3. ¿Qué técnica de la Parte D mejoró más la generalización?**

(escribe aquí)